## Problema 2 — Regressão Logística

### 1. Carregamento e Exploração dos Dados

In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=45)

X = dataset.data.features.copy()
y = dataset.data.targets.copy()

df = pd.concat([X, y], axis=1)

display(df.head())

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [2]:
print("Dimensões:", df.shape)

print("\nInformações gerais:")
df.info()

print("\nValores ausentes:")
print(df.isnull().sum())

print("\nEstatísticas descritivas:")
display(df.describe())

Dimensões: (303, 14)

Informações gerais:
<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 33.3 KB

Valores ausentes:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
t

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,299.000000,301.000000,303.000000
mean,54.438944,0.679868,3.158416,131.689769,246.693069,0.148515,0.990099,149.607261,0.326733,1.039604,1.600660,0.672241,4.734219,0.937294
std,9.038662,0.467299,0.960126,17.599748,51.776918,0.356198,0.994971,22.875003,0.469794,1.161075,0.616226,0.937438,1.939706,1.228536
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000,0.000000
25%,48.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,133.500000,0.000000,0.000000,1.000000,0.000000,3.000000,0.000000
50%,56.000000,1.000000,3.000000,130.000000,241.000000,0.000000,1.000000,153.000000,0.000000,0.800000,2.000000,0.000000,3.000000,0.000000
75%,61.000000,1.000000,4.000000,140.000000,275.000000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,1.000000,7.000000,2.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000,4.000000


In [ ]:
# Distribuição original dos diagnósticos
print("Classes originais:")
print(df["num"].value_counts().sort_index())

# Binariza a variável alvo
df["target"] = (df["num"] > 0).astype(int)

# Distribuição das classes binárias
print("\nClasses após binarização:")
print(df["target"].value_counts().sort_index())

Classes originais:
num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64

Classes após binarização:
target
0    164
1    139
Name: count, dtype: int64


## 2. Pré-processamento dos Dados

### 2.1. Separação das variáveis

In [ ]:
# Separa as variáveis independentes
X = df.drop(columns=["num", "target"])

# Seleciona a variável alvo binarizada
y = df["target"].copy()

variaveis_numericas = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak",
    "ca"
]

variaveis_categoricas = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "thal"
]

print("Variáveis independentes:", X.shape[1])
print("Variável alvo:", y.name)

print("\nVariáveis numéricas:")
print(variaveis_numericas)

print("\nVariáveis categóricas:")
print(variaveis_categoricas)

Variáveis independentes: 13
Variável alvo: target

Variáveis numéricas:
['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']

Variáveis categóricas:
['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']


### 2.2. Divisão dos dados em treino e teste

In [5]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Amostras de treino:", X_treino.shape)
print("Amostras de teste:", X_teste.shape)

print("\nDistribuição no treino:")
print(y_treino.value_counts(normalize=True))

print("\nDistribuição no teste:")
print(y_teste.value_counts(normalize=True))

Amostras de treino: (242, 13)
Amostras de teste: (61, 13)

Distribuição no treino:
target
0    0.541322
1    0.458678
Name: proportion, dtype: float64

Distribuição no teste:
target
0    0.540984
1    0.459016
Name: proportion, dtype: float64


### 2.3. Tratamento dos valores ausentes e padronização

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pipeline_numerico = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("padronizacao", StandardScaler())
])

pipeline_categorico = Pipeline([
    ("imputacao", SimpleImputer(strategy="most_frequent")),
    ("codificacao", OneHotEncoder(handle_unknown="ignore"))
])

# Combina as transformações
preprocessador = ColumnTransformer(
    transformers=[
        ("numericas", pipeline_numerico, variaveis_numericas),
        ("categoricas", pipeline_categorico, variaveis_categoricas)
    ],
    sparse_threshold=0.0
)

X_treino_proc = preprocessador.fit_transform(X_treino)

X_teste_proc = preprocessador.transform(X_teste)

print("Formato do treino:", X_treino_proc.shape)
print("Formato do teste:", X_teste_proc.shape)

Formato do treino: (242, 25)
Formato do teste: (61, 25)


### 2.4. Verificação dos dados processados

In [ ]:
print("Valores ausentes no treino:",
      np.isnan(X_treino_proc).sum())

print("Valores ausentes no teste:",
      np.isnan(X_teste_proc).sum())

print("\nDimensões finais:")
print("Treino:", X_treino_proc.shape)
print("Teste:", X_teste_proc.shape)

# Recupera os nomes dos atributos transformados
nomes_atributos = preprocessador.get_feature_names_out()

print("\nQuantidade de atributos:", len(nomes_atributos))

print("\nAtributos após o pré-processamento:")
print(nomes_atributos)

Valores ausentes no treino: 0
Valores ausentes no teste: 0

Dimensões finais:
Treino: (242, 25)
Teste: (61, 25)

Quantidade de atributos: 25

Atributos após o pré-processamento:
['numericas__age' 'numericas__trestbps' 'numericas__chol'
 'numericas__thalach' 'numericas__oldpeak' 'numericas__ca'
 'categoricas__sex_0.0' 'categoricas__sex_1.0' 'categoricas__cp_1.0'
 'categoricas__cp_2.0' 'categoricas__cp_3.0' 'categoricas__cp_4.0'
 'categoricas__fbs_0.0' 'categoricas__fbs_1.0' 'categoricas__restecg_0.0'
 'categoricas__restecg_1.0' 'categoricas__restecg_2.0'
 'categoricas__exang_0.0' 'categoricas__exang_1.0'
 'categoricas__slope_1.0' 'categoricas__slope_2.0'
 'categoricas__slope_3.0' 'categoricas__thal_3.0' 'categoricas__thal_6.0'
 'categoricas__thal_7.0']


## 3. Treinamento do Modelo

### 3.1. Implementação da Regressão Logística

In [8]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(
    solver="liblinear",
    random_state=42,
    max_iter=1000
)

modelo.fit(X_treino_proc, y_treino)

print("Treinamento concluído!")

Treinamento concluído!


### 3.2. Verificação da Convergência

In [9]:
# Número de iterações realizadas
iteracoes = modelo.n_iter_[0]

print("Iterações realizadas:", iteracoes)
print("Máximo permitido:", modelo.max_iter)

if iteracoes < modelo.max_iter:
    print("O modelo convergiu antes do limite de iterações.")
else:
    print("Verifique se houve aviso de não convergência.")

Iterações realizadas: 6
Máximo permitido: 1000
O modelo convergiu antes do limite de iterações.


### 3.3. Previsões no Conjunto de Teste

In [10]:
# Prevê as classes dos pacientes
y_pred = modelo.predict(X_teste_proc)

# Calcula a probabilidade da classe 1
y_prob = modelo.predict_proba(X_teste_proc)[:, 1]

comparacao = pd.DataFrame({
    "Diagnóstico Real": y_teste.to_numpy(),
    "Diagnóstico Previsto": y_pred,
    "Probabilidade de Doença": y_prob
})

display(comparacao.head(10))

,Diagnóstico Real,Diagnóstico Previsto,Probabilidade de Doença
0,0,0,0.215407
1,0,1,0.582650
2,0,0,0.024364
3,0,0,0.037389
4,0,0,0.333411
5,0,0,0.015759
6,0,0,0.170164
7,0,0,0.242934
8,1,1,0.755861
9,0,0,0.374332
